# Lab 04 — SCD Type 1 Product Dimension

This notebook builds a **current-state product dimension** from the validated Silver transaction table and maintains it with a Delta Lake `MERGE`.

SCD Type 1 overwrites changed attributes in place. It is appropriate when only the latest value matters or when correcting inaccurate data. It does **not** preserve the previous description or price.

## Objectives

- Derive one deterministic current product row per `stock_code`.
- Create a stable product surrogate key.
- Seed a Type 1 Delta dimension without duplicating existing products.
- Generate a small, controlled attribute-change batch.
- Apply overwrite behavior with an idempotent `MERGE`.
- Validate uniqueness, overwrite results, replay safety, and Delta history.


## 1. Load shared configuration

Run the shared configuration so this notebook uses the same catalog, schema, volume, batch identifier, and table names as the earlier Lab 4 notebooks.

In [0]:
%run ./lab04_00_config

# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


runtime_selection,contract_name,yaml_version,governance_status,column_count,supersedes,runtime_selected
v1,online_retail,1,active,8,null,true


Runtime configuration ready: dbr_dev.parvinbadalov
Volume: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Runtime contract: online_retail v1 (governance status: active)
Schema policy: fail


In [0]:
import sys

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

lab04_root = (
    "/Workspace/Users/parvinbadalov@yahoo.com/"
    "Databricks-Academy-Lakehouse/labs/lab_04_silver_quality"
)

if lab04_root not in sys.path:
    sys.path.append(lab04_root)

from src.merge_utils import (
    add_record_hash,
    merge_scd_type1,
)

product_scd1_table = table_names["product_scd1"]
scd1_change_batch_id = f"{batch_id}_scd1_change"
scd1_change_path = (
    f"{paths['scd_changes']}/type1/batch_id={scd1_change_batch_id}"
)

print(f"Silver source: {silver_table}")
print(f"SCD Type 1 target: {product_scd1_table}")
print(f"Controlled change path: {scd1_change_path}")
print("Reusable MERGE module: src.merge_utils")


Silver source: dbr_dev.parvinbadalov.lab04_silver_transactions
SCD Type 1 target: dbr_dev.parvinbadalov.lab04_product_scd1
Controlled change path: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/scd_changes/type1/batch_id=initial_scd1_change
Reusable MERGE module: src.merge_utils


## 2. Verify the Silver transaction source

The product dimension must be derived only from the analytics-ready Silver table. The notebook stops early if notebook 04 has not created or populated that table.

In [0]:
if not spark.catalog.tableExists(silver_table):
    raise RuntimeError(
        f"Required Silver table does not exist: {silver_table}. "
        "Run lab04_04_silver_merge.ipynb first."
    )

silver_transactions_df = spark.table(silver_table)
silver_transaction_count = silver_transactions_df.count()
if silver_transaction_count == 0:
    raise ValueError(f"Silver transaction table is empty: {silver_table}")

required_source_columns = {
    "transaction_line_id", "invoice_no", "stock_code", "description",
    "unit_price", "invoice_timestamp", "source_record_hash",
    "source_batch_id", "silver_updated_at",
}
missing_source_columns = sorted(required_source_columns - set(silver_transactions_df.columns))
if missing_source_columns:
    raise AssertionError(f"Silver source is missing columns: {missing_source_columns}")

print(f"✅ Silver source ready with {silver_transaction_count:,} transactions.")


✅ Silver source ready with 315,101 transactions.


## 3. Derive one current record per product

A transaction table can contain many rows for the same product. We select the latest observation using a deterministic ordering:

1. newest invoice timestamp,
2. newest Silver update timestamp,
3. transaction line ID as a stable tie-breaker.

`product_sk` is a stable hash of `stock_code`. `source_record_hash` is a product-level hash used to detect attribute changes without rewriting unchanged rows.

In [0]:
latest_product_window = (
    Window.partitionBy("stock_code")
    .orderBy(
        F.col("invoice_timestamp").desc(),
        F.col("silver_updated_at").desc(),
        F.col("transaction_line_id").desc(),
    )
)

latest_product_rows_df = (
    silver_transactions_df
    .filter(
        F.col("stock_code").isNotNull()
        & (F.length(F.trim("stock_code")) > 0)
    )
    .withColumn(
        "_product_rank",
        F.row_number().over(latest_product_window),
    )
    .filter(F.col("_product_rank") == 1)
)

product_source_df = (
    latest_product_rows_df
    .select(
        F.sha2(F.col("stock_code"), 256).alias("product_sk"),
        F.col("stock_code"),
        F.col("description"),
        F.col("unit_price")
        .cast("decimal(18,4)")
        .alias("latest_observed_price"),
        F.col("invoice_no").alias("source_invoice_no"),
        F.col("transaction_line_id").alias(
            "source_transaction_line_id"
        ),
        F.col("invoice_timestamp").alias(
            "source_event_timestamp"
        ),
        F.col("source_batch_id"),
    )
)

product_source_df = add_record_hash(
    product_source_df,
    [
        "stock_code",
        "description",
        "latest_observed_price",
    ],
    output_column="source_record_hash",
)

product_source_count = product_source_df.count()
product_source_distinct = (
    product_source_df.select("stock_code").distinct().count()
)

if (
    product_source_count == 0
    or product_source_count != product_source_distinct
):
    raise AssertionError(
        f"Product source key validation failed: "
        f"rows={product_source_count}, "
        f"distinct stock codes={product_source_distinct}."
    )

print(f"Derived {product_source_count:,} unique products.")
display(product_source_df.orderBy("stock_code").limit(30))


Derived 3,628 unique products.


product_sk,stock_code,description,latest_observed_price,source_invoice_no,source_transaction_line_id,source_event_timestamp,source_batch_id,source_record_hash
56acd2f85500d246e0851f77035f8b4414a57c4f31735c8f37c9cec0f3739d47,10002,INFLATABLE POLITICAL GLOBE,0.8500,550452,5a5ad332b65c0ef0ba84a9fea5d9465f467333ac50995e57fd61d316de648084,2011-04-18T12:56:00.000Z,initial,101f81f320fbbde205a982f4d76faae2ec9873a1e60a2b11c3ad29666f39e061
2ab4e55c7fdf332c78e4f36966a981f452ffb6758026085a4ad9df565a1df82d,10080,GROOVY CACTUS INFLATABLE,0.3900,577773,574af94ff0b2803d98464b20759fa5e04e2750a5067e962ea300718d523ee275,2011-11-21T15:57:00.000Z,initial,362dafe771dbedd66c5328ea4b72748759622674b15e74bf2b0d497e077c7e9c
801aaa199f1030bd990a6bd8da7e1bd76f1461bee56880924ccbd49f1166178b,10120,DOGGY RUBBER,0.2100,580502,34c037f99e74e8dc9b0ab51a0910044bf6a0d4502c6677d44d33169ccd1b3a5b,2011-12-04T13:15:00.000Z,initial,2bc8de476d7e9e68ae6f5c83a14a960a3dc9c5f530b6cf156f92853ecc63a5d1
c603961ad4629b9a249a64757df4c2d63249ad4babd6c908a97bcc679368ee8d,10123C,HEARTS WRAPPING TAPE,0.6500,548491,43881e426937a68d6c103c10d9f652c92e1d9342ac7195ba06a28d20418ce3e4,2011-03-31T13:14:00.000Z,initial,f90770ecaa1c4629e4bd8c1d287e02049ca21d7bb26cf00ffdbe481ff2ecb994
6a84538adbe548d147f2d20d9e6df55d9ed7ff53e4a4326f695eccb031fc13ab,10124A,SPOTS ON RED BOOKCOVER TAPE,0.4200,574686,d49304baf4e6d457d5ea5afc5d0921b5e376930e75e239b51a26acab784661e7,2011-11-06T13:00:00.000Z,initial,15ca5847e977cb6a108ef084967826d56aff15672373d1d7cb4f55e64a91b5bf
63d621285bb94218c6676555283e19fa7bb61719ec56c2416255164a73fb88ed,10124G,ARMY CAMO BOOKCOVER TAPE,0.4200,574686,ef20dd2506d762c7113ddd959d29fdc1d5a0e060a121e305d7fdb151dc2e4f0c,2011-11-06T13:00:00.000Z,initial,2738f8d05e6885a4e8bae18914c3bb439da25e4396d86ebe881abb4fd5772636
7cb3fc848fa0e795b7b15cbef125090b271d93b387645020e9788219e2fda761,10125,MINI FUNKY DESIGN TAPES,0.8500,581494,e95fc1f9f1bb16338a42fbecdeb751fec3d47f83a262c4b9ff831d48bc579a06,2011-12-09T10:13:00.000Z,initial,8bf93d52c48e03395788df46878986406141be8d53a02dbb6650e41be8585603
7d686424482e81086e0e399c1f1bd9173551b38dcf2221902c9c067876fd4e56,10133,COLOURING PENCILS BROWN TUBE,0.4200,565541,ec543ed93e763d4d54cf3d58125d9a6d07a857b9e1e6939df3d6d1898ba5c8d6,2011-09-05T12:00:00.000Z,initial,2ee072ab0fd7dc2c222e770aece8b2cb32b72fa33c8b0379bcea19e2de5a0938
9190a41c01288f3725256e28f95cfd7679d185ad55d42ad5a1e2a5d7c2b091e3,10135,COLOURING PENCILS BROWN TUBE,2.4600,580727,6cd902c1d48299d517ecd3f4f0315e779311670d8258df1c3e1b20472e163c37,2011-12-05T17:17:00.000Z,initial,ae4fc1c31cf134345d55f99f2a30caed21832329ad11e399a98d42821a63f65c
72b209306c1a1031b9b3dbec63cf58b0beabbf3e3f0a40c63ef5df093b62dfb8,11001,ASSTD DESIGN RACING CAR PEN,3.2900,580727,6b08b9cf60eae708bdf74155f28fc34dce782461ecdbf92da42663b4d4e6e715,2011-12-05T17:17:00.000Z,initial,213f38231320d1e90cf743a4dd2e9f01219ad82544fbb4d1aa4b802466dcb3a7


## 4. Verify the pre-created SCD Type 1 target

The SCD Type 1 table structure is created by **`lab04_00_setup`** outside the production Job.

This notebook only validates that the structure exists before applying current-state overwrite logic.


In [0]:
if reset_demo_objects:
    raise ValueError(
        "reset_demo_objects=true is not allowed inside the production Job. "
        "Run lab04_00_setup manually for a clean structural rebuild."
    )

if not spark.catalog.tableExists(product_scd1_table):
    raise RuntimeError(
        f"Required SCD Type 1 table does not exist: {product_scd1_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

required_scd1_columns = {
    "product_sk",
    "stock_code",
    "description",
    "latest_observed_price",
    "source_invoice_no",
    "source_transaction_line_id",
    "source_event_timestamp",
    "source_record_hash",
    "source_batch_id",
    "effective_from",
    "effective_to",
    "is_current",
    "created_at",
    "updated_at",
}

missing_scd1_columns = sorted(
    required_scd1_columns - set(spark.table(product_scd1_table).columns)
)

if missing_scd1_columns:
    raise AssertionError(
        "Pre-created SCD Type 1 target is missing columns: "
        + ", ".join(missing_scd1_columns)
    )

print(f"✅ Pre-created SCD Type 1 table is ready: {product_scd1_table}")


✅ Pre-created SCD Type 1 table is ready: dbr_dev.parvinbadalov.lab04_product_scd1


## 5. Seed products that are not already present

The baseline load inserts only missing product keys. It deliberately does not overwrite existing rows. This lets the later controlled change batch demonstrate the overwrite action clearly and makes the entire notebook safe to rerun.

In [0]:
target_keys_before_df = spark.table(product_scd1_table).select("stock_code")
expected_seed_inserts = product_source_df.join(target_keys_before_df, "stock_code", "left_anti").count()
target_count_before_seed = spark.table(product_scd1_table).count()

seed_insert_values = {
    "product_sk": "source.product_sk",
    "stock_code": "source.stock_code",
    "description": "source.description",
    "latest_observed_price": "source.latest_observed_price",
    "source_invoice_no": "source.source_invoice_no",
    "source_transaction_line_id": "source.source_transaction_line_id",
    "source_event_timestamp": "source.source_event_timestamp",
    "source_record_hash": "source.source_record_hash",
    "source_batch_id": "source.source_batch_id",
    "effective_from": "source.source_event_timestamp",
    "effective_to": "CAST(NULL AS TIMESTAMP)",
    "is_current": "true",
    "created_at": "current_timestamp()",
    "updated_at": "current_timestamp()",
}

(
    DeltaTable.forName(spark, product_scd1_table).alias("target")
    .merge(product_source_df.alias("source"), "target.stock_code = source.stock_code")
    .whenNotMatchedInsert(values=seed_insert_values)
    .execute()
)

target_count_after_seed = spark.table(product_scd1_table).count()
actual_seed_growth = target_count_after_seed - target_count_before_seed
if actual_seed_growth != expected_seed_inserts:
    raise AssertionError(
        f"Seed mismatch: expected {expected_seed_inserts} inserts, observed {actual_seed_growth}."
    )

print(f"Products before seed: {target_count_before_seed:,}")
print(f"Products inserted: {actual_seed_growth:,}")
print(f"Products after seed: {target_count_after_seed:,}")


Products before seed: 3,628
Products inserted: 0
Products after seed: 3,628


## 6. Generate a controlled product-change batch

The public workbook does not provide a formal product-master change feed, so the lab creates a deterministic test batch for three products:

- append ` [SCD1 UPDATED]` to the description,
- increase the observed price by `1.0000`,
- calculate a new product-level hash,
- persist the batch under `test_data/scd_changes/type1`.

The test data is separate from source and production-style tables. Re-running this cell replaces only the same controlled Lab 4 test batch.

In [0]:
change_sample_size = min(3, product_source_count)
if change_sample_size == 0:
    raise ValueError(
        "No products are available for the SCD Type 1 change test."
    )

scd1_change_df = (
    product_source_df
    .orderBy("stock_code")
    .limit(change_sample_size)
    .withColumn(
        "description",
        F.concat(
            F.regexp_replace(
                F.coalesce(
                    F.col("description"),
                    F.lit("UNKNOWN PRODUCT"),
                ),
                r" \[SCD1 UPDATED\]$",
                "",
            ),
            F.lit(" [SCD1 UPDATED]"),
        ),
    )
    .withColumn(
        "latest_observed_price",
        (
            F.coalesce(
                F.col("latest_observed_price"),
                F.lit(0).cast("decimal(18,4)"),
            )
            + F.lit(1).cast("decimal(18,4)")
        ).cast("decimal(18,4)"),
    )
    .withColumn(
        "source_batch_id",
        F.lit(scd1_change_batch_id),
    )
    .drop("source_record_hash")
)

scd1_change_df = add_record_hash(
    scd1_change_df,
    [
        "stock_code",
        "description",
        "latest_observed_price",
    ],
    output_column="source_record_hash",
)

(
    scd1_change_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(scd1_change_path)
)

persisted_change_df = (
    spark.read.format("delta").load(scd1_change_path)
)

if persisted_change_df.count() != change_sample_size:
    raise AssertionError(
        "Persisted SCD Type 1 test batch count does not match "
        "the generated batch."
    )

print(
    f"✅ Controlled change batch written with "
    f"{change_sample_size} products."
)
display(persisted_change_df.orderBy("stock_code"))


✅ Controlled change batch written with 3 products.


product_sk,stock_code,description,latest_observed_price,source_invoice_no,source_transaction_line_id,source_event_timestamp,source_batch_id,source_record_hash
56acd2f85500d246e0851f77035f8b4414a57c4f31735c8f37c9cec0f3739d47,10002,INFLATABLE POLITICAL GLOBE [SCD1 UPDATED],1.8500,550452,5a5ad332b65c0ef0ba84a9fea5d9465f467333ac50995e57fd61d316de648084,2011-04-18T12:56:00.000Z,initial_scd1_change,3bab8c81b3914370fd456abe24a3447550136d5ada65bf9ff14294f4c4ff6fd2
2ab4e55c7fdf332c78e4f36966a981f452ffb6758026085a4ad9df565a1df82d,10080,GROOVY CACTUS INFLATABLE [SCD1 UPDATED],1.3900,577773,574af94ff0b2803d98464b20759fa5e04e2750a5067e962ea300718d523ee275,2011-11-21T15:57:00.000Z,initial_scd1_change,56489c587ca9c824eedac79a2b9a2e89152baf379fb6b22dac043427aa564e04
801aaa199f1030bd990a6bd8da7e1bd76f1461bee56880924ccbd49f1166178b,10120,DOGGY RUBBER [SCD1 UPDATED],1.2100,580502,34c037f99e74e8dc9b0ab51a0910044bf6a0d4502c6677d44d33169ccd1b3a5b,2011-12-04T13:15:00.000Z,initial_scd1_change,f20e0b49bdf8781c6cbaa608c503f6d6b25f6018e8a6243e86248034a037b985


## 7. Classify the Type 1 changes

The merge plan distinguishes inserts, updates, and unchanged rows before changing the target. On the first execution, the controlled products should be updates. On an exact rerun, they should be unchanged.

In [0]:
target_before_change_df = spark.table(product_scd1_table).select(
    F.col("stock_code").alias("target_stock_code"),
    F.col("description").alias("previous_description"),
    F.col("latest_observed_price").alias("previous_price"),
    F.col("source_record_hash").alias("target_record_hash"),
)

change_classification_df = (
    persisted_change_df.alias("source")
    .join(
        target_before_change_df.alias("target"),
        F.col("source.stock_code") == F.col("target.target_stock_code"),
        "left",
    )
    .select(
        F.col("source.stock_code").alias("stock_code"),
        F.col("source.description").alias("incoming_description"),
        F.col("source.latest_observed_price").alias("incoming_price"),
        F.col("target.previous_description"),
        F.col("target.previous_price"),
        F.when(F.col("target.target_stock_code").isNull(), F.lit("INSERT"))
        .when(
            ~F.col("source.source_record_hash").eqNullSafe(F.col("target.target_record_hash")),
            F.lit("UPDATE"),
        )
        .otherwise(F.lit("UNCHANGED"))
        .alias("merge_action"),
    )
)

change_plan_df = (
    change_classification_df.groupBy("merge_action")
    .agg(F.count("*").alias("rows"))
    .orderBy("merge_action")
)
change_plan = {row["merge_action"]: row["rows"] for row in change_plan_df.collect()}
expected_change_inserts = change_plan.get("INSERT", 0)
expected_change_updates = change_plan.get("UPDATE", 0)
expected_change_unchanged = change_plan.get("UNCHANGED", 0)

display(change_plan_df)
display(
    change_classification_df.select(
        "stock_code", "merge_action", "previous_description",
        "incoming_description", "previous_price", "incoming_price",
    ).orderBy("stock_code")
)


merge_action,rows
UNCHANGED,3


stock_code,merge_action,previous_description,incoming_description,previous_price,incoming_price
10002,UNCHANGED,INFLATABLE POLITICAL GLOBE [SCD1 UPDATED],INFLATABLE POLITICAL GLOBE [SCD1 UPDATED],1.8500,1.8500
10080,UNCHANGED,GROOVY CACTUS INFLATABLE [SCD1 UPDATED],GROOVY CACTUS INFLATABLE [SCD1 UPDATED],1.3900,1.3900
10120,UNCHANGED,DOGGY RUBBER [SCD1 UPDATED],DOGGY RUBBER [SCD1 UPDATED],1.2100,1.2100


## 8. Apply the SCD Type 1 MERGE

Matched rows are updated only when the product hash changes. The merge overwrites the current description and price while preserving `product_sk` and `created_at`. No historical row is inserted for the previous values.

In [0]:
scd1_merge_source_df = (
    persisted_change_df
    .withColumn("effective_from", F.current_timestamp())
    .withColumn(
        "effective_to",
        F.lit(None).cast("timestamp"),
    )
    .withColumn("is_current", F.lit(True))
    .withColumn("created_at", F.current_timestamp())
)

scd1_insert_columns = [
    "product_sk",
    "stock_code",
    "description",
    "latest_observed_price",
    "source_invoice_no",
    "source_transaction_line_id",
    "source_event_timestamp",
    "source_record_hash",
    "source_batch_id",
    "effective_from",
    "effective_to",
    "is_current",
    "created_at",
]

scd1_mutable_columns = [
    "description",
    "latest_observed_price",
    "source_invoice_no",
    "source_transaction_line_id",
    "source_event_timestamp",
    "source_batch_id",
    "effective_from",
    "effective_to",
    "is_current",
]

target_count_before_change = spark.table(
    product_scd1_table
).count()

merge_scd_type1(
    scd1_merge_source_df,
    product_scd1_table,
    business_key="stock_code",
    mutable_columns=scd1_mutable_columns,
    insert_columns=scd1_insert_columns,
    hash_column="source_record_hash",
    updated_at_column="updated_at",
)

target_count_after_change = spark.table(
    product_scd1_table
).count()

actual_change_growth = (
    target_count_after_change - target_count_before_change
)

if actual_change_growth != expected_change_inserts:
    raise AssertionError(
        f"Change MERGE count mismatch: expected "
        f"{expected_change_inserts} inserts, "
        f"observed growth {actual_change_growth}."
    )

print(f"Planned SCD1 updates: {expected_change_updates:,}")
print(
    f"Planned unchanged rows: "
    f"{expected_change_unchanged:,}"
)
print(f"Target row growth: {actual_change_growth:,}")
print(
    "✅ SCD Type 1 MERGE completed through "
    "src.merge_utils.merge_scd_type1()."
)


Planned SCD1 updates: 0
Planned unchanged rows: 3
Target row growth: 0
✅ SCD Type 1 MERGE completed through src.merge_utils.merge_scd_type1().


## 9. Prove that old values were overwritten

The comparison below shows the before and after attributes for the controlled products. The target contains the new values only; the earlier attributes do not remain as separate dimension rows.

In [0]:
target_after_change_df = (
    spark.table(product_scd1_table)
    .select(
        F.col("stock_code").alias("current_stock_code"),
        F.col("description").alias("current_description"),
        F.col("latest_observed_price").alias("current_price"),
        F.col("source_record_hash").alias("current_record_hash"),
        F.col("source_batch_id").alias("current_batch_id"),
        F.col("created_at"),
        F.col("updated_at"),
    )
)

# Evidence for the action classification in THIS execution.
overwrite_evidence_df = (
    change_classification_df.alias("plan")
    .join(
        target_after_change_df.alias("current"),
        F.col("plan.stock_code")
        == F.col("current.current_stock_code"),
        "inner",
    )
    .select(
        F.col("plan.stock_code"),
        F.col("plan.merge_action"),
        F.col("plan.previous_description"),
        F.col("current.current_description"),
        F.col("plan.previous_price"),
        F.col("current.current_price"),
        F.col("current.current_batch_id"),
        F.col("current.created_at"),
        F.col("current.updated_at"),
    )
)

# Stable SCD1 proof even after the notebook is replayed:
# compare the original Silver-derived product state with current SCD1 state.
controlled_keys_df = (
    persisted_change_df.select("stock_code").distinct()
)

scd1_before_after_df = (
    controlled_keys_df
    .join(
        product_source_df.select(
            "stock_code",
            F.col("description").alias("before_description"),
            F.col("latest_observed_price").alias("before_price"),
        ),
        "stock_code",
    )
    .join(
        spark.table(product_scd1_table).select(
            "stock_code",
            F.col("description").alias("after_description"),
            F.col("latest_observed_price").alias("after_price"),
            F.col("source_batch_id").alias("current_batch_id"),
        ),
        "stock_code",
    )
    .withColumn(
        "scd1_result",
        F.when(
            (
                ~F.col("before_description")
                .eqNullSafe(F.col("after_description"))
            )
            | (
                ~F.col("before_price")
                .eqNullSafe(F.col("after_price"))
            ),
            F.lit("OVERWRITTEN"),
        ).otherwise(F.lit("UNCHANGED")),
    )
    .select(
        "stock_code",
        "before_description",
        "after_description",
        "before_price",
        "after_price",
        "scd1_result",
        "current_batch_id",
    )
    .orderBy("stock_code")
)

mismatch_count = (
    persisted_change_df.alias("expected")
    .join(
        target_after_change_df.alias("actual"),
        F.col("expected.stock_code")
        == F.col("actual.current_stock_code"),
        "left",
    )
    .filter(
        F.col("actual.current_stock_code").isNull()
        | ~F.col("expected.source_record_hash")
        .eqNullSafe(F.col("actual.current_record_hash"))
    )
    .count()
)

if mismatch_count != 0:
    raise AssertionError(
        f"SCD Type 1 overwrite validation found "
        f"{mismatch_count} mismatches."
    )

display(overwrite_evidence_df.orderBy("stock_code"))
display(scd1_before_after_df)

print(
    "✅ Controlled products contain only the latest attributes; "
    "stable before/after evidence is available in scd1_before_after_df."
)


stock_code,merge_action,previous_description,current_description,previous_price,current_price,current_batch_id,created_at,updated_at
10002,UNCHANGED,INFLATABLE POLITICAL GLOBE [SCD1 UPDATED],INFLATABLE POLITICAL GLOBE [SCD1 UPDATED],1.8500,1.8500,initial_scd1_change,2026-08-09T21:04:38.380Z,2026-08-09T21:04:50.910Z
10080,UNCHANGED,GROOVY CACTUS INFLATABLE [SCD1 UPDATED],GROOVY CACTUS INFLATABLE [SCD1 UPDATED],1.3900,1.3900,initial_scd1_change,2026-08-09T21:04:38.380Z,2026-08-09T21:04:50.910Z
10120,UNCHANGED,DOGGY RUBBER [SCD1 UPDATED],DOGGY RUBBER [SCD1 UPDATED],1.2100,1.2100,initial_scd1_change,2026-08-09T21:04:38.380Z,2026-08-09T21:04:50.910Z


stock_code,before_description,after_description,before_price,after_price,scd1_result,current_batch_id
10002,INFLATABLE POLITICAL GLOBE,INFLATABLE POLITICAL GLOBE [SCD1 UPDATED],0.8500,1.8500,OVERWRITTEN,initial_scd1_change
10080,GROOVY CACTUS INFLATABLE,GROOVY CACTUS INFLATABLE [SCD1 UPDATED],0.3900,1.3900,OVERWRITTEN,initial_scd1_change
10120,DOGGY RUBBER,DOGGY RUBBER [SCD1 UPDATED],0.2100,1.2100,OVERWRITTEN,initial_scd1_change


✅ Controlled products contain only the latest attributes; stable before/after evidence is available in scd1_before_after_df.


## 10. Validate current-state dimension quality

A Type 1 dimension must contain exactly one row per business key. All rows are current, no end timestamp is used, and core identifiers must not be null.

In [0]:
scd1_df = spark.table(product_scd1_table)
scd1_quality_df = scd1_df.agg(
    F.count("*").alias("dimension_rows"),
    F.countDistinct("stock_code").alias("distinct_stock_codes"),
    F.sum(F.col("stock_code").isNull().cast("long")).alias("null_stock_codes"),
    F.sum(F.col("product_sk").isNull().cast("long")).alias("null_product_keys"),
    F.sum((~F.col("is_current")).cast("long")).alias("non_current_rows"),
    F.sum(F.col("effective_to").isNotNull().cast("long")).alias("closed_rows"),
)
quality = scd1_quality_df.first().asDict()
if quality["dimension_rows"] != quality["distinct_stock_codes"]:
    raise AssertionError(f"SCD1 duplicate business keys found: {quality}")
if any(quality[name] != 0 for name in ["null_stock_codes", "null_product_keys", "non_current_rows", "closed_rows"]):
    raise AssertionError(f"SCD1 current-state quality rules failed: {quality}")

display(scd1_quality_df)
print("✅ SCD Type 1 uniqueness and current-state checks passed.")


dimension_rows,distinct_stock_codes,null_stock_codes,null_product_keys,non_current_rows,closed_rows
3628,3628,0,0,0,0


✅ SCD Type 1 uniqueness and current-state checks passed.


## 11. Replay the same change batch

The same controlled batch is merged again. Because the hashes already match, the conditional update must skip every row and the target count must remain unchanged.

In [0]:
count_before_replay = spark.table(
    product_scd1_table
).count()

merge_scd_type1(
    scd1_merge_source_df,
    product_scd1_table,
    business_key="stock_code",
    mutable_columns=scd1_mutable_columns,
    insert_columns=scd1_insert_columns,
    hash_column="source_record_hash",
    updated_at_column="updated_at",
)

count_after_replay = spark.table(
    product_scd1_table
).count()

remaining_change_count = (
    persisted_change_df.alias("source")
    .join(
        spark.table(product_scd1_table)
        .select(
            F.col("stock_code").alias("target_stock_code"),
            F.col("source_record_hash").alias(
                "target_record_hash"
            ),
        )
        .alias("target"),
        F.col("source.stock_code")
        == F.col("target.target_stock_code"),
        "left",
    )
    .filter(
        F.col("target.target_stock_code").isNull()
        | ~F.col("source.source_record_hash")
        .eqNullSafe(F.col("target.target_record_hash"))
    )
    .count()
)

if (
    count_after_replay != count_before_replay
    or remaining_change_count != 0
):
    raise AssertionError(
        f"SCD1 replay failed: before={count_before_replay}, "
        f"after={count_after_replay}, "
        f"remaining changes={remaining_change_count}."
    )

print(
    "✅ Idempotency passed through merge_scd_type1(): "
    "replay inserted 0 rows and left 0 pending changes."
)
print(f"Dimension row count remains {count_after_replay:,}.")


✅ Idempotency passed through merge_scd_type1(): replay inserted 0 rows and left 0 pending changes.
Dimension row count remains 3,628.


## 12. Review Delta history and final results

Delta history records the merge operations. The dimension table still exposes only the latest product values, which is the defining limitation and benefit of Type 1.

In [0]:
scd1_history_df = spark.sql(f"DESCRIBE HISTORY {product_scd1_table}")
display(
    scd1_history_df.select(
        "version", "timestamp", "operation", "operationParameters", "operationMetrics",
    ).orderBy(F.col("version").desc()).limit(10)
)

final_results_df = spark.createDataFrame(
    [
        ("silver_transaction_rows", silver_transaction_count),
        ("unique_product_source_rows", product_source_count),
        ("seed_inserts_this_run", actual_seed_growth),
        ("controlled_change_rows", change_sample_size),
        ("planned_change_updates", expected_change_updates),
        ("planned_change_unchanged", expected_change_unchanged),
        ("dimension_rows", count_after_replay),
        ("remaining_changes_after_replay", remaining_change_count),
    ],
    ["validation", "result"],
)
display(final_results_df)
print("✅ SCD Type 1 notebook completed successfully.")


version,timestamp,operation,operationParameters,operationMetrics
19,2026-08-10T20:32:20.000Z,MERGE,"Map(predicate -> [""(stock_code#30165 = stock_code#28351)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#30171 <=> source_record_hash#28358)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3219, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1131, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2050)"
18,2026-08-10T20:32:07.000Z,MERGE,"Map(predicate -> [""(stock_code#28777 = stock_code#28351)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#28783 <=> source_record_hash#28358)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2970, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1045, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1888)"
17,2026-08-10T20:31:55.000Z,MERGE,"Map(predicate -> [""(stock_code#27439 = stock_code#26812)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2178, materializeSourceTimeMs -> 8, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3628, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2141)"
16,2026-08-10T20:21:16.000Z,MERGE,"Map(predicate -> [""(stock_code#20638 = stock_code#18824)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#20644 <=> source_record_hash#18831)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3477, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1285, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySour

validation,result
silver_transaction_rows,315101
unique_product_source_rows,3628
seed_inserts_this_run,0
controlled_change_rows,3
planned_change_updates,0
planned_change_unchanged,3
dimension_rows,3628
remaining_changes_after_replay,0


✅ SCD Type 1 notebook completed successfully.


## Evidence to capture

Save screenshots of:

1. the unique product source preview,
2. the INSERT / UPDATE / UNCHANGED merge plan,
3. the before-versus-current overwrite comparison,
4. the uniqueness and current-state quality result,
5. the idempotent replay message,
6. Delta history and the final results table.
